In [ ]:
# %% ════════════════════════════════════════════════════════════════════════
# CALM-FORGE TRAIN — huấn luyện Gemma 4 12B cho đồ án fuzz, một file chạy thẳng.
#
# Chạy 2-GPU Blackwell (Snowflake, RTX PRO 6000, OFFLINE):
#     torchrun --standalone --nproc_per_node=2 calm_gemma4_train_blackwell.py
# Chạy 1-GPU (smoke/Colab):
#     python calm_gemma4_train_blackwell.py
#
# Gộp 3 nguồn:
#   (A) Bài học xương máu từ blackwell/train_iterations (≈195 cell lặp tay):
#       - Gemma 4 đa phương thức → PHẢI đóng băng vision/audio tower, nếu không
#         DDP báo "unused parameters" và chết.
#       - Unsloth từ chối config gemma4 khi offline → vá passthrough cho
#         `_Gemma4KVSharedSafeProxy.__getattr__`.
#       - completion-only loss (mask prompt = -100) — chỉ học phần code sinh ra.
#       - r64/alpha128 + rsLoRA, gradient-checkpointing=True (bản DDP-an-toàn,
#         KHÔNG dùng "unsloth" khi chạy DDP).
#       - Bug lớn nhất họ mất nhiều giờ: lora_B đứng yên ≈ 0, model KHÔNG học.
#         → cài chốt tự kiểm |lora_B| sống, abort sớm nếu không nhúc nhích.
#       - Mỗi rank chỉ thấy 1 GPU; tránh evaluate() chỉ-rank0 (gây deadlock DDP).
#   (B) Best-practice Unsloth/LoRA 2026 (tra cứu mạng, trích dẫn cuối file):
#       rsLoRA r64/a128 cho 12B; LR cosine thấp + early-stop cho data nhỏ;
#       eff-batch ~16–32; weight_decay 0.01; max_grad_norm 1.0; warmup 3–10%;
#       cảnh báo overfit khi train-loss < 0.2.
#   (C) Đột phá riêng cho đồ án: ngoài eval token-loss, có FUZZ-GEN EVAL — sau
#       train, model tự sinh seed từ prompt giữ-lại rồi ĐO ĐÚNG KPI đồ án:
#       % parse được (ast), % import framework, % chứa phép so vi sai/assert.
#       Token-loss lái việc dừng sớm; fuzz-gen đo "model có thực sự biết sinh
#       seed tự-phán-quyết" — thứ token-loss không thấy.
#
# Dữ liệu: dùng gemma4_sft_final/ (cell 1+2 đã dựng). Mỗi record có
# instruction/input/output + text render template Gemma + nhãn axis (8 trục
# PAX/HELIX) + source (github_issue | axiom_synthetic).
# ════════════════════════════════════════════════════════════════════════════
import ast, json, os, sys, time, glob, math, datetime
from pathlib import Path

# ── Cấu hình (env-override được; mặc định khớp Blackwell 2-GPU 96GB) ─────────
def E(k, d): return os.environ.get(k, d)
CFG = {
    "model_dir":   E("MODEL_DIR", "/tmp/gemma4_12b_base"),     # thư mục model Gemma 4 12B
    "data_dir":    E("DATA_DIR", ""),                           # rỗng → tự dò gemma4_sft_final
    "out_dir":     E("OUT_DIR", "/tmp/calm_gemma4_adapter"),
    "max_len":     int(E("MAX_LEN", "2048")),
    "lora_r":      int(E("LORA_R", "64")),
    "lora_alpha":  int(E("LORA_ALPHA", "128")),
    "lora_dropout":float(E("LORA_DROPOUT", "0.05")),  # data nhỏ → có dropout chống overfit
    "use_rslora":  E("USE_RSLORA", "1") == "1",
    "load_4bit":   E("LOAD_4BIT", "0") == "1",        # 12B trên 96GB: bf16 dư sức, để 0
    "epochs":      float(E("EPOCHS", "3")),
    "lr":          float(E("LR", "1e-5")),            # thấp: 12B + ~2.3k mẫu, ưu tiên tổng quát
    "per_device_batch": int(E("PER_DEVICE_BATCH", "4")),
    "grad_accum":  int(E("GRAD_ACCUM", "4")),         # eff batch = 4×4×nGPU
    "weight_decay":float(E("WEIGHT_DECAY", "0.01")),
    "warmup_ratio":float(E("WARMUP_RATIO", "0.05")),
    "max_grad_norm":float(E("MAX_GRAD_NORM", "1.0")),
    "neftune":     float(E("NEFTUNE", "5")),          # nhiễu nhúng — quý cho data nhỏ
    "eval_ratio":  float(E("EVAL_RATIO", "0.06")),    # nếu thiếu val.jsonl thì tách từ train
    "early_stop_patience": int(E("EARLY_STOP", "3")),
    "loraB_check_step": int(E("LORAB_CHECK_STEP", "20")),
    "fuzz_gen_eval": E("FUZZ_GEN_EVAL", "1") == "1",
    "ddp_find_unused": E("DDP_FIND_UNUSED", "0") == "1",  # đã đóng băng tower → False nhanh hơn
    "web_preflight": E("WEB_PREFLIGHT", "0") == "1",      # chỉ chạy nơi CÓ internet (Colab staging)
    "seed": int(E("SEED", "20260611")),
}
RANK = int(E("RANK", E("LOCAL_RANK", "0")))
WORLD = int(E("WORLD_SIZE", "1"))
IS_MAIN = RANK == 0
def log(*a):
    if IS_MAIN: print(f"[{datetime.datetime.now():%H:%M:%S}] [r{RANK}/{WORLD}]", *a, flush=True)

# ════════════════════════════════════════════════════════════════════════════
# PHẦN TORCH-FREE: dựng prompt/completion + thống kê (test được không cần GPU)
# ════════════════════════════════════════════════════════════════════════════
RESP_MARK = "<start_of_turn>model\n"

def find_data_dir():
    if CFG["data_dir"] and (Path(CFG["data_dir"]) / "train.jsonl").is_file():
        return Path(CFG["data_dir"])
    for d in [".", r"C:\Users\fagon\Loxi\th6\11", "/content", "/tmp",
              "/tmp/gemma4_full_support/dataset"]:
        for sub in ("gemma4_sft_final", "gemma4_sft_out", ""):
            p = Path(d) / sub if sub else Path(d)
            if (p / "train.jsonl").is_file():
                return p
    raise SystemExit("Không thấy train.jsonl — đặt DATA_DIR trỏ tới gemma4_sft_final/")

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def build_prompt_completion(rec, tok=None):
    """Tách record thành (prompt, completion, axis, source).
    prompt = mọi thứ tới hết RESP_MARK; completion = code + <end_of_turn>.
    Ưu tiên dựng lại từ instruction/input/output để không phụ thuộc render cũ;
    nếu có tokenizer thì dùng apply_chat_template cho khớp template gốc của Gemma."""
    instr = (rec.get("instruction") or "").strip()
    inp = (rec.get("input") or "").strip()
    out = (rec.get("output") or "").strip()
    user_msg = f"{instr}\n\n{inp}".strip() if inp else instr
    if not out and rec.get("text"):                       # fallback: bóc từ text render sẵn
        t = rec["text"]
        if RESP_MARK in t:
            head, tail = t.split(RESP_MARK, 1)
            return head + RESP_MARK, tail, rec.get("axis", "op_core"), rec.get("source", "?")
    completion = f"```python\n{out}```<end_of_turn>\n"
    if tok is not None and hasattr(tok, "apply_chat_template"):
        try:
            prompt = tok.apply_chat_template([{"role": "user", "content": user_msg}],
                                             tokenize=False, add_generation_prompt=True)
            return prompt, completion, rec.get("axis", "op_core"), rec.get("source", "?")
        except Exception:
            pass
    prompt = f"<start_of_turn>user\n{user_msg}<end_of_turn>\n{RESP_MARK}"
    return prompt, completion, rec.get("axis", "op_core"), rec.get("source", "?")

def axis_stats(recs):
    from collections import Counter
    return dict(Counter(r.get("axis", "?") for r in recs).most_common()), \
           dict(Counter(r.get("source", "?") for r in recs).most_common())

# ── Tra cứu mạng: CHỈ chạy nơi có internet (máy staging Colab), không bắt buộc ─
WEB_GROUNDING = """\
Hyperparameter được neo theo tài liệu 2026 (biên dịch sẵn vào CFG, không gọi mạng lúc train):
- Unsloth LoRA guide: rsLoRA r64/alpha128 cho lớp 12B; lora_dropout 0–0.1 (0 nhanh nhất,
  >0 chống overfit cho data nhỏ); target = 7 proj q/k/v/o/gate/up/down.
- LR: 2e-4 mặc định LoRA, nhưng data nhỏ + model lớn → hạ về 1e-5..2e-5 + cosine, warmup 3–10%.
- Epoch 1–3, early-stop; weight_decay 0.01; max_grad_norm 1.0.
- Cảnh báo overfit: train-loss < 0.2 = đang học thuộc lòng.
- Unsloth DDP: torchrun --nproc_per_node=N, tự bật khi >1 GPU; per_device_batch nhỏ + grad_accum.
"""
def web_preflight():
    """Tùy chọn: chạy ở nơi CÓ internet để in best-practice mới nhất trước khi
    bê script sang Blackwell offline. Không bao giờ làm gãy train nếu offline."""
    if not CFG["web_preflight"]:
        return
    try:
        import urllib.request
        url = "https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide"
        with urllib.request.urlopen(url, timeout=8) as r:
            ok = r.status == 200
        log(f"[web] với tới Unsloth docs: {'OK' if ok else 'fail'} — đối chiếu CFG trước khi stage.")
    except Exception as e:
        log(f"[web] offline/không với được ({type(e).__name__}) — dùng grounding biên dịch sẵn.")
    log(WEB_GROUNDING)

# ════════════════════════════════════════════════════════════════════════════
# PHẦN CẦN TORCH/UNSLOTH: chạy khi thực train
# ════════════════════════════════════════════════════════════════════════════
def main():
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    web_preflight()

    import torch
    # — Vá proxy gemma4 của Unsloth (offline) trước khi import nặng —
    try:
        import unsloth  # noqa
        import unsloth_zoo.temporary_patches.gemma4 as g4
        P = getattr(g4, "_Gemma4KVSharedSafeProxy", None)
        if P is not None:
            P.__getattr__ = lambda self, n: getattr(object.__getattribute__(self, "_real"), n)
            log("đã vá _Gemma4KVSharedSafeProxy.__getattr__ → passthrough")
    except Exception as e:
        log("proxy gemma4 warn:", repr(e))

    from unsloth import FastLanguageModel
    from datasets import Dataset
    from transformers import TrainerCallback, EarlyStoppingCallback
    from trl import SFTTrainer, SFTConfig

    if torch.cuda.is_available():
        torch.cuda.set_device(int(E("LOCAL_RANK", "0")))
    log(f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} "
        f"| world={WORLD} | model={CFG['model_dir']}")

    # ── Model + LoRA ────────────────────────────────────────────────────────
    model, tok = FastLanguageModel.from_pretrained(
        model_name=CFG["model_dir"], max_seq_length=CFG["max_len"],
        load_in_4bit=CFG["load_4bit"], load_in_16bit=not CFG["load_4bit"],
        full_finetuning=False, local_files_only=True)
    tok = getattr(tok, "tokenizer", tok)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # gradient-checkpointing=True (DDP-an-toàn), KHÔNG "unsloth" khi đa GPU
    gc_mode = True if WORLD > 1 else "unsloth"
    model = FastLanguageModel.get_peft_model(
        model, r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
        bias="none", use_rslora=CFG["use_rslora"], use_gradient_checkpointing=gc_mode,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        random_state=CFG["seed"])

    # Đóng băng vision/audio (text-only) — nếu không DDP "unused params" sẽ chết
    froze = 0
    for n, p in model.named_parameters():
        if any(k in n for k in ("vision_tower", "audio_tower", "embed_vision", "embed_audio")):
            if p.requires_grad: froze += 1
            p.requires_grad_(False)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"đóng băng {froze} tham số đa-phương-thức | trainable={trainable/1e6:.1f}M")

    # Tham số canh chừng: 1 lora_B của lớp ngôn ngữ — phải nhúc nhích khỏi 0
    watch_B = next(p for n, p in model.named_parameters() if "lora_B" in n and p.requires_grad)

    # ── Dữ liệu → prompt/completion, mask completion-only ────────────────────
    ddir = find_data_dir()
    train_recs = read_jsonl(ddir / "train.jsonl")
    val_path = ddir / "val.jsonl"
    if val_path.is_file():
        eval_recs = read_jsonl(val_path)
    else:                                       # tự tách nếu thiếu split eval
        k = max(8, int(len(train_recs) * CFG["eval_ratio"]))
        eval_recs, train_recs = train_recs[:k], train_recs[k:]
    ax, sc = axis_stats(train_recs)
    log(f"train={len(train_recs)} eval={len(eval_recs)} từ {ddir}")
    log(f"axis: {ax}"); log(f"source: {sc}")

    eos = tok.eos_token_id
    def to_features(recs):
        feats = []
        for r in recs:
            prompt, completion, _, _ = build_prompt_completion(r, tok)
            pid = tok(prompt, add_special_tokens=True)["input_ids"]
            cid = tok(completion, add_special_tokens=False)["input_ids"] + [eos]
            ids = (pid + cid)[:CFG["max_len"]]
            labels = ([-100] * len(pid) + cid)[:CFG["max_len"]]   # mask prompt
            if len(set(labels)) == 1:           # toàn -100 (prompt tràn max_len) → bỏ
                continue
            feats.append({"input_ids": ids, "labels": labels,
                          "attention_mask": [1] * len(ids), "n_tokens": len(ids)})
        return feats
    train_ds = Dataset.from_list(to_features(train_recs))
    eval_ds = Dataset.from_list(to_features(eval_recs))

    def collate(batch):
        m = max(len(b["input_ids"]) for b in batch)
        pad = tok.pad_token_id
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for b in batch:
            g = m - len(b["input_ids"])
            out["input_ids"].append(b["input_ids"] + [pad] * g)
            out["attention_mask"].append(b["attention_mask"] + [0] * g)
            out["labels"].append(b["labels"] + [-100] * g)
        return {k: torch.tensor(v) for k, v in out.items()}

    # ── Chốt tự kiểm: lora_B có sống không (bài học xương máu) ───────────────
    class LoraBLiveness(TrainerCallback):
        def __init__(self): self.b0 = float(watch_B.detach().abs().sum())
        def on_step_end(self, args, state, control, **kw):
            if state.global_step == CFG["loraB_check_step"]:
                now = float(watch_B.detach().abs().sum())
                delta = abs(now - self.b0)
                log(f"[liveness] step{state.global_step} |B|: {self.b0:.3e} → {now:.3e} (Δ={delta:.3e})")
                if delta < 1e-9:
                    control.should_training_stop = True
                    log("‼ lora_B KHÔNG nhúc nhích — model không học. Dừng để khỏi đốt giờ GPU vô ích.")
            return control

    eff = CFG["per_device_batch"] * CFG["grad_accum"] * max(WORLD, 1)
    steps_per_epoch = max(1, len(train_ds) // eff)
    log(f"eff_batch={eff} | ~{steps_per_epoch} step/epoch | ~{int(steps_per_epoch*CFG['epochs'])} step tổng")

    args = SFTConfig(
        output_dir=CFG["out_dir"], dataset_kwargs={"skip_prepare_dataset": True},
        per_device_train_batch_size=CFG["per_device_batch"],
        per_device_eval_batch_size=CFG["per_device_batch"],
        gradient_accumulation_steps=CFG["grad_accum"],
        num_train_epochs=CFG["epochs"], learning_rate=CFG["lr"],
        lr_scheduler_type="cosine", warmup_ratio=CFG["warmup_ratio"],
        weight_decay=CFG["weight_decay"], max_grad_norm=CFG["max_grad_norm"],
        bf16=True, fp16=False, gradient_checkpointing=(WORLD > 1),
        logging_steps=10, eval_strategy="steps", eval_steps=max(25, steps_per_epoch // 2),
        save_strategy="steps", save_steps=max(25, steps_per_epoch // 2), save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        group_by_length=True, length_column_name="n_tokens",
        neftune_noise_alpha=(CFG["neftune"] or None),
        ddp_find_unused_parameters=CFG["ddp_find_unused"],
        report_to="none", seed=CFG["seed"], remove_unused_columns=False)

    trainer = SFTTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=collate,
        callbacks=[LoraBLiveness(),
                   EarlyStoppingCallback(early_stopping_patience=CFG["early_stop_patience"])])

    log("▶ bắt đầu train …")
    t0 = time.time()
    trainer.train()
    log(f"✔ train xong trong {(time.time()-t0)/60:.1f} phút")

    # ── Lưu adapter (chỉ rank0) + xác nhận lora_B != 0 trong FILE ────────────
    if IS_MAIN:
        Path(CFG["out_dir"]).mkdir(parents=True, exist_ok=True)
        model.save_pretrained(CFG["out_dir"]); tok.save_pretrained(CFG["out_dir"])
        bsum = float(watch_B.detach().abs().sum())
        log(f"đã lưu adapter → {CFG['out_dir']} | |lora_B|={bsum:.3e} "
            f"({'OK đã học' if bsum > 1e-6 else '‼ B≈0'})")

    # ── FUZZ-GEN EVAL: đo KPI đồ án (rank0, sau train → an toàn DDP) ─────────
    if IS_MAIN and CFG["fuzz_gen_eval"]:
        fuzz_gen_eval(model, tok, eval_recs[:24])

def fuzz_gen_eval(model, tok, recs):
    """Sinh seed từ prompt giữ-lại rồi đo: parse được? import framework? có
    assert/so-vi-sai? — KPI 'model có biết sinh seed tự-phán-quyết' của đồ án."""
    import torch, re
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    rows, n_parse, n_imp, n_assert = [], 0, 0, 0
    for r in recs:
        prompt, _, axis, source = build_prompt_completion(r, tok)
        ids = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=320, do_sample=False,
                                  pad_token_id=tok.pad_token_id)
        gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        m = re.search(r"```(?:python)?\n(.*?)```", gen, re.S)
        code = m.group(1) if m else gen
        ok_parse = False
        try: ast.parse(code); ok_parse = True
        except Exception: pass
        ok_imp = bool(re.search(r"import torch|import tensorflow|from torch|from tensorflow", code))
        ok_assert = bool(re.search(r"assert|allclose|assert_close|isnan|isinf|equal\(", code))
        n_parse += ok_parse; n_imp += ok_imp; n_assert += ok_assert
        rows.append({"axis": axis, "source": source, "parse": ok_parse,
                     "import": ok_imp, "self_check": ok_assert, "code_head": code[:160]})
    N = max(len(rows), 1)
    summary = {"n": len(rows), "parse_rate": n_parse / N, "import_rate": n_imp / N,
               "self_check_rate": n_assert / N, "samples": rows}
    p = Path(CFG["out_dir"]) / "fuzz_gen_eval.json"
    p.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    log(f"[FUZZ-KPI] parse={summary['parse_rate']:.0%} "
        f"import={summary['import_rate']:.0%} self-check={summary['self_check_rate']:.0%} "
        f"(n={len(rows)}) → {p}")

# ════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    if "--selftest" in sys.argv:
        # Kiểm phần torch-free trên dataset thật, không cần GPU/torch.
        ddir = find_data_dir()
        recs = read_jsonl(ddir / "train.jsonl")
        bad = 0
        for r in recs:
            p, c, ax, so = build_prompt_completion(r)
            if RESP_MARK not in p or "```" not in c: bad += 1
        ax, sc = axis_stats(recs)
        print(f"[selftest] data={ddir} records={len(recs)} build_lỗi={bad}")
        print(f"[selftest] axis={ax}")
        print(f"[selftest] source={sc}")
        print(f"[selftest] ví dụ prompt[:120]={build_prompt_completion(recs[0])[0][:120]!r}")
        print(f"[selftest] ví dụ completion[:80]={build_prompt_completion(recs[0])[1][:80]!r}")
        print("[selftest] OK" if bad == 0 else "[selftest] CÓ LỖI")
    else:
        main()

# ── Nguồn tra cứu (06/2026) ──────────────────────────────────────────────────
# - Unsloth LoRA Hyperparameters Guide: https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide
# - Unsloth Multi-GPU DDP: https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth/ddp
# - Unsloth Gemma 3 finetune: https://unsloth.ai/blog/gemma3
# - Fine-tuning hyperparameter best practices: https://latitude.so/blog/fine-tuning-llms-hyperparameter-best-practices


In [ ]:
import os, sys, subprocess, shutil, json
print('=== PYTHON ===', sys.version.split()[0])
print('=== nvidia-smi ===')
try:
    print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
except Exception as e:
    print('no nvidia-smi', e)
for p in ['torch','unsloth','unsloth_zoo','datasets','transformers','trl','peft','accelerate','bitsandbytes']:
    try:
        m=__import__(p); print(f'OK   {p:14s}', getattr(m,'__version__','?'))
    except Exception as e:
        print(f'MISS {p:14s}', type(e).__name__)
try:
    import torch; print('cuda avail', torch.cuda.is_available(), 'n_gpu', torch.cuda.device_count())
except Exception as e:
    print('torch import fail', e)
try:
    from snowflake.snowpark.context import get_active_session
    session=get_active_session(); print('SNOWPARK session OK', session.get_current_account())
except Exception as e:
    print('snowpark session', e)
print('=== disk /tmp ==='); print(subprocess.run(['df','-h','/tmp'],capture_output=True,text=True).stdout)

In [ ]:
import os, tarfile, subprocess
from snowflake.snowpark.context import get_active_session
session = get_active_session()
STAGE='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_12b_it_unsloth_full_quality_pack_20260610'
os.makedirs('/tmp/support_dl', exist_ok=True)
print('downloading support pack...', flush=True)
session.file.get(f'{STAGE}/support/gemma4_12b_it_support_pack.tar.gz', '/tmp/support_dl')
print('got:', os.listdir('/tmp/support_dl'), flush=True)
os.makedirs('/tmp/support', exist_ok=True)
with tarfile.open('/tmp/support_dl/gemma4_12b_it_support_pack.tar.gz') as t:
    t.extractall('/tmp/support')
print('=== extracted tree (depth<=2) ===', flush=True)
for root,dirs,files in os.walk('/tmp/support'):
    depth=root[len('/tmp/support'):].count('/')
    if depth>2: continue
    for f in sorted(files):
        fp=os.path.join(root,f)
        print(f'{os.path.getsize(fp):>12d}  {fp}')

In [ ]:
for p in ['/tmp/support/meta/RUNBOOK_gemma4_12b_it_pack.md','/tmp/support/meta/model_manifest.json','/tmp/support/scripts/SNOWFLAKE_INSTALL_AND_PROBE_GEMMA4_12B_IT.py']:
    print('\n'+'='*30+' '+p+' '+'='*30)
    print(open(p).read())
import os
print('\n=== wheels list ===')
print('\n'.join(sorted(os.listdir('/tmp/support/wheels'))))

In [ ]:
print(open('/tmp/support/scripts/SNOWFLAKE_INSTALL_AND_PROBE_GEMMA4_12B_IT.py').read())

In [ ]:
import subprocess, sys
cmd=[sys.executable,'-m','pip','install','--no-index','--find-links=/tmp/support/wheels',
     'unsloth','unsloth_zoo','trl','bitsandbytes','xformers','cut_cross_entropy']
print('RUN:',' '.join(cmd),flush=True)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line.rstrip(),flush=True)
rc=p.wait()
print('PIP EXIT',rc,flush=True)
assert rc==0,'pip failed'
print('=== verify imports ===',flush=True)
for pkg in ['unsloth','unsloth_zoo','trl','bitsandbytes','xformers']:
    try:
        m=__import__(pkg); print('OK',pkg,getattr(m,'__version__','?'))
    except Exception as e:
        print('FAIL',pkg,repr(e))

In [ ]:
import subprocess,sys
r=subprocess.run([sys.executable,'-m','pip','download','--no-deps','-d','/tmp/_pt','tyro'],capture_output=True,text=True,timeout=120)
print('rc',r.returncode)
print(r.stdout[-1500:])
print(r.stderr[-1500:])

In [ ]:
import os,glob,subprocess,sys
for d in ['/opt/wheels','/tmp/support/wheels']:
    print('===',d,'===')
    print('\n'.join(sorted(os.listdir(d))) if os.path.isdir(d) else 'MISSING')
print('=== find tyro anywhere ===')
print(subprocess.run(['bash','-c','find / -iname "tyro*" 2>/dev/null | head -40'],capture_output=True,text=True).stdout or 'none')
print('=== is tyro importable already? ===')
try:
    import tyro; print('tyro present', tyro.__version__)
except Exception as e:
    print('no tyro:', e)

In [ ]:
import subprocess, sys
WH='/tmp/support/wheels'
def pip(*args):
    cmd=[sys.executable,'-m','pip','install','--no-index',f'--find-links={WH}',*args]
    print('RUN:',' '.join(cmd),flush=True)
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for l in p.stdout: print(l.rstrip(),flush=True)
    rc=p.wait(); print('EXIT',rc,flush=True); return rc
# 1) matched-set deps (lets transformers/hub/etc align)
rc1=pip('trl','bitsandbytes')
# 2) unsloth core without its tyro dep
rc2=pip('--no-deps','unsloth','unsloth_zoo','cut_cross_entropy')
print('=== import test ===',flush=True)
import importlib
for pkg in ['transformers','trl','bitsandbytes','unsloth_zoo','unsloth']:
    try:
        importlib.invalidate_caches()
        m=importlib.import_module(pkg); print('OK',pkg,getattr(m,'__version__','?'))
    except Exception as e:
        print('FAIL',pkg,repr(e)[:300])

In [ ]:
import subprocess, sys
WH='/tmp/support/wheels'
r=subprocess.run([sys.executable,'-m','pip','install','--no-index',f'--find-links={WH}','trl','bitsandbytes'],capture_output=True,text=True)
print('--- pip tail ---')
print('\n'.join([l for l in r.stdout.splitlines() if 'Successfully' in l or 'Installing' in l or 'error' in l.lower()]) or r.stdout[-1200:])
print('rc',r.returncode)
print('--- pip show ---')
print(subprocess.run([sys.executable,'-m','pip','show','trl','bitsandbytes'],capture_output=True,text=True).stdout)
print('--- fresh subprocess import (mimics torchrun) ---')
chk='import trl,bitsandbytes,peft,datasets,transformers,torch;print("trl",trl.__version__,"bnb",bitsandbytes.__version__,"tf",transformers.__version__,"torch",torch.__version__,"gpus",torch.cuda.device_count())'
r2=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('STDOUT',r2.stdout)
print('STDERR',r2.stderr[-800:])

In [ ]:
import subprocess, sys
WH='/tmp/support/wheels'
r=subprocess.run([sys.executable,'-m','pip','install','--no-index',f'--find-links={WH}','--no-deps','trl','bitsandbytes'],capture_output=True,text=True)
print([l for l in r.stdout.splitlines() if 'Successfully' in l] or r.stdout[-800:])
print('rc',r.returncode, r.stderr[-400:])
chk='import trl,bitsandbytes;print("trl",trl.__version__,"bnb",bitsandbytes.__version__)'
r2=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('IMPORT STDOUT:',r2.stdout.strip())
print('IMPORT STDERR:',r2.stderr[-600:])

In [ ]:
import subprocess, sys
chk=r'''
import os
os.environ["UNSLOTH_DISABLE_AUTO_UPDATES"]="1"
import unsloth
from unsloth import FastModel, FastLanguageModel
import unsloth_zoo, trl, peft, bitsandbytes, cut_cross_entropy, torch
print("IMPORTS_OK gpus", torch.cuda.device_count(), "unsloth", unsloth.__version__)
'''
r=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('STDOUT:\n',r.stdout[-2000:])
print('STDERR:\n',r.stderr[-2500:])

In [ ]:
import os, time, shutil
from snowflake.snowpark.context import get_active_session
session=get_active_session()
STAGE='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_12b_it_unsloth_full_quality_pack_20260610'
MODEL_DIR='/tmp/gemma4_12b_base'
shutil.rmtree(MODEL_DIR,ignore_errors=True); os.makedirs(MODEL_DIR,exist_ok=True)
t0=time.time()
print('downloading model_snapshot/* (22GB) ...',flush=True)
res=session.file.get(f'{STAGE}/model_snapshot/', MODEL_DIR)
print(f'downloaded {len(res)} files in {time.time()-t0:.0f}s',flush=True)
tot=0
for f in sorted(os.listdir(MODEL_DIR)):
    sz=os.path.getsize(os.path.join(MODEL_DIR,f)); tot+=sz
    print(f'{sz:>13d}  {f}')
print(f'TOTAL {tot/1024**3:.2f} GB')

In [ ]:
import subprocess, sys, os
SCRIPT='/tmp/support/scripts/calm_gemma4_train_blackwell.py'
DATA='/tmp/support/dataset'
MODEL='/tmp/gemma4_12b_base'
print('dataset files:', os.listdir(DATA), flush=True)
env=dict(os.environ, MODEL_DIR=MODEL, DATA_DIR=DATA)
r=subprocess.run([sys.executable, SCRIPT, '--selftest'], capture_output=True, text=True, env=env)
print('rc', r.returncode)
print('STDOUT:\n', r.stdout[-2500:])
print('STDERR:\n', r.stderr[-1500:])

In [ ]:
import subprocess, sys, os, time
LAUNCHER='/tmp/run_train_patched.py'
LOG='/tmp/train.log'
envv={
 'MODEL_DIR':'/tmp/gemma4_12b_base',
 'DATA_DIR':'/tmp/support/dataset',
 'OUT_DIR':'/tmp/calm_gemma4_12b_it_adapter',
 'MAX_LEN':'2048','PER_DEVICE_BATCH':'4','GRAD_ACCUM':'4',
 'LORA_R':'64','LORA_ALPHA':'128','LR':'1e-5','EPOCHS':'3',
 'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','TOKENIZERS_PARALLELISM':'false',
}
env_prefix=' '.join(f'{k}={v}' for k,v in envv.items())
py=sys.executable
cmd=f'cd /tmp && {env_prefix} nohup {py} -m torch.distributed.run --standalone --nproc_per_node=2 {LAUNCHER} > {LOG} 2>&1 &'
open(LOG,'w').close()
print('LAUNCH:\n', cmd, flush=True)
subprocess.Popen(['bash','-lc',cmd], start_new_session=True)
print('launched; waiting 45s for model load + first step...', flush=True)
time.sleep(45)
print('=== running procs ===', flush=True)
print(subprocess.run(['bash','-lc','ps -eo pid,etime,cmd | grep -E "distributed.run|run_train_patched" | grep -v grep'],capture_output=True,text=True).stdout or 'NONE')
print('=== train.log tail ===', flush=True)
print(subprocess.run(['bash','-lc',f'tail -n 35 {LOG}'],capture_output=True,text=True).stdout)

In [ ]:
import os, subprocess, sys
WH='/tmp/support/wheels'
import glob
print('relevant wheels:')
for pat in ['transformers*','huggingface_hub*','tokenizers*','safetensors*','hf_xet*','hf_transfer*','datasets*']:
    print(' ', [os.path.basename(x) for x in glob.glob(f'{WH}/{pat}')])
pkgs=['transformers','huggingface_hub','safetensors','hf_xet','hf_transfer','datasets']
r=subprocess.run([sys.executable,'-m','pip','install','--no-index',f'--find-links={WH}','--no-deps','--upgrade',*pkgs],capture_output=True,text=True)
print([l for l in r.stdout.splitlines() if 'Successfully' in l] or r.stdout[-1000:])
print('rc',r.returncode, r.stderr[-500:])
chk=r'import transformers,tokenizers,huggingface_hub;print("tf",transformers.__version__,"tok",tokenizers.__version__,"hub",huggingface_hub.__version__);import unsloth;from unsloth import FastModel;print("unsloth+tf OK")'
r2=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('VERIFY STDOUT:',r2.stdout[-1200:])
print('VERIFY STDERR:',r2.stderr[-1500:])

In [ ]:
import subprocess, sys
chk=r'''
import sys
print("--- sys.path ---")
for p in sys.path: print(p)
import pyarrow; print("pyarrow", pyarrow.__version__, pyarrow.__file__)
try:
    import datasets; print("datasets OK", datasets.__version__, datasets.__file__)
except Exception as e:
    print("datasets FAIL", repr(e)[:200])
import importlib.util, glob, os
print("--- datasets locations ---")
for base in sys.path:
    d=os.path.join(base,"datasets","__init__.py")
    if os.path.exists(d): print(d)
'''
r=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print(r.stdout[-2500:])
print('ERR',r.stderr[-800:])

In [ ]:
import glob, os, subprocess, sys
WH='/tmp/support/wheels'
print('pyarrow/arrow wheels:', [os.path.basename(x) for x in glob.glob(f'{WH}/*arrow*')+glob.glob(f'{WH}/*pyarrow*')])
print('ALL wheels:')
print('\n'.join(sorted(os.path.basename(x) for x in glob.glob(f'{WH}/*.whl'))))

In [ ]:
import subprocess, sys
r=subprocess.run([sys.executable,'-m','pip','uninstall','-y','datasets'],capture_output=True,text=True)
print(r.stdout[-500:], r.stderr[-300:])
chk=r'''
import datasets; print("datasets", datasets.__version__, datasets.__file__)
import pyarrow; print("pyarrow", pyarrow.__version__)
import transformers; print("transformers", transformers.__version__)
import unsloth
from unsloth import FastModel
import trl, peft, bitsandbytes
print("FULL CHAIN OK")
'''
r2=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('STDOUT:',r2.stdout[-1500:])
print('STDERR:',r2.stderr[-1800:])

In [ ]:
import subprocess, sys
WH='/tmp/support/wheels'
r=subprocess.run([sys.executable,'-m','pip','install','--no-index',f'--find-links={WH}','--no-deps','--upgrade','peft'],capture_output=True,text=True)
print([l for l in r.stdout.splitlines() if 'Successfully' in l] or r.stdout[-400:], 'rc',r.returncode)
chk=r'''
import datasets, transformers, peft
print("datasets",datasets.__version__,"transformers",transformers.__version__,"peft",peft.__version__,"peft@",peft.__file__)
import unsloth
from unsloth import FastModel
import trl, bitsandbytes, cut_cross_entropy, torch
print("FULL CHAIN OK gpus", torch.cuda.device_count())
'''
r2=subprocess.run([sys.executable,'-c',chk],capture_output=True,text=True)
print('STDOUT:',r2.stdout[-1500:])
print('STDERR:',r2.stderr[-1800:])

In [ ]:
import subprocess
print('=== process alive? ===')
print(subprocess.run(['bash','-lc','ps -eo pid,etime,cmd | grep -E "distributed.run|calm_gemma4" | grep -v grep'],capture_output=True,text=True).stdout or 'NO training process (finished or died)')
print('=== GPU util / mem ===')
print(subprocess.run(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used,memory.total','--format=csv'],capture_output=True,text=True).stdout)
print('=== last 45 lines of /tmp/train.log ===')
print(subprocess.run(['bash','-lc','tail -n 45 /tmp/train.log'],capture_output=True,text=True).stdout)

In [ ]:
src=open('/tmp/support/scripts/calm_gemma4_train_blackwell.py').read().splitlines()
for i,l in enumerate(src,1):
    if any(k in l for k in ['import unsloth','from unsloth','from trl','import trl','SFTConfig(','SFTTrainer(']) or (380<=i<=475):
        print(f'{i:4d}| {l}')

In [ ]:
src=open('/tmp/support/scripts/calm_gemma4_train_blackwell.py').read().splitlines()
for i,l in enumerate(src,1):
    if 405<=i<=434:
        print(f'{i:4d}| {l}')

In [ ]:
wrapper = r'''
import os, sys, runpy, inspect
import unsloth          # patch first for optimizations
import unsloth_zoo      # noqa
import trl

def patch_init(cls, rename=None, _seen=set()):
    if cls in _seen or cls is object: return
    _seen.add(cls)
    orig = cls.__dict__.get("__init__", None)
    if orig is not None:
        try:
            sig = inspect.signature(orig)
            params = sig.parameters
            has_var_kw = any(p.kind == p.VAR_KEYWORD for p in params.values())
            valid = set(params)
            def newinit(self, *a, __orig=orig, __valid=valid, __hvk=has_var_kw, __nm=cls.__name__, **kw):
                if rename:
                    for old, new in rename.items():
                        if old in kw and new not in kw:
                            kw[new] = kw.pop(old)
                if not __hvk:
                    dropped = [k for k in list(kw) if k not in __valid]
                    for k in dropped: kw.pop(k)
                    if dropped: print(f"[compat] {__nm} dropped: {dropped}", flush=True)
                return __orig(self, *a, **kw)
            newinit.__signature__ = sig
            cls.__init__ = newinit
        except (ValueError, TypeError):
            pass

# patch every class in the MRO of SFTConfig/SFTTrainer (covers Unsloth subclass + real trl + TrainingArguments)
for base in list(trl.SFTConfig.__mro__):
    patch_init(base)
for base in list(trl.SFTTrainer.__mro__):
    patch_init(base, rename={"tokenizer": "processing_class"})
print("[compat] patched MRO of SFTConfig & SFTTrainer", flush=True)

TARGET = "/tmp/support/scripts/calm_gemma4_train_blackwell.py"
sys.argv = [TARGET]
runpy.run_path(TARGET, run_name="__main__")
'''
open('/tmp/run_train_patched.py','w').write(wrapper)
print('wrote /tmp/run_train_patched.py', len(wrapper),'bytes')
import shutil; shutil.rmtree('/tmp/unsloth_compiled_cache', ignore_errors=True); print('cleared unsloth_compiled_cache')

In [ ]:
import subprocess
print(subprocess.run(['bash','-lc',"grep -nE 'Error|Exception|Traceback|compat|raise |assert|CUDA|out of memory|not supported' /tmp/train.log | head -40"],capture_output=True,text=True).stdout)
print('================ context around first Error ================')
print(subprocess.run(['bash','-lc',"ln=$(grep -nE 'Error|Exception' /tmp/train.log | head -1 | cut -d: -f1); start=$((ln-25)); [ $start -lt 1 ] && start=1; sed -n \"${start},$((ln+5))p\" /tmp/train.log"],capture_output=True,text=True).stdout)

In [ ]:
import subprocess
print(subprocess.run(['bash','-lc',"grep -nE 'Error|Exception|Traceback|compat|raise |assert|CUDA|out of memory|not supported' /tmp/train.log | head -40"],capture_output=True,text=True).stdout)
print('================ context around first Error ================')
print(subprocess.run(['bash','-lc',"ln=$(grep -nE 'Error|Exception' /tmp/train.log | head -1 | cut -d: -f1); start=$((ln-25)); [ $start -lt 1 ] && start=1; sed -n \"${start},$((ln+5))p\" /tmp/train.log"],capture_output=True,text=True).stdout)

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
print('--- per-process VRAM ---')
print(subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_memory','--format=csv,noheader'],capture_output=True,text=True).stdout)

In [ ]:
import subprocess, time, re, statistics as st
Q='index,utilization.gpu,utilization.memory,memory.used,power.draw,power.limit,temperature.gpu,clocks.sm,clocks.mem'
samples={0:[],1:[]}
pwr={0:[],1:[]}
print('sampling 12s @ ~0.5s ...')
for _ in range(24):
    out=subprocess.run(['nvidia-smi',f'--query-gpu={Q}','--format=csv,noheader,nounits'],capture_output=True,text=True).stdout.strip().splitlines()
    for ln in out:
        c=[x.strip() for x in ln.split(',')]
        g=int(c[0]); samples[g].append(float(c[1])); pwr[g].append(float(c[4]))
    time.sleep(0.5)
for g in (0,1):
    u=samples[g]; p=pwr[g]
    print(f'GPU{g}: util avg={st.mean(u):.0f}% max={max(u):.0f}% min={min(u):.0f}% | power avg={st.mean(p):.0f}W max={max(p):.0f}W')
print('\n=== full snapshot ===')
print(subprocess.run(['nvidia-smi',f'--query-gpu={Q}','--format=csv'],capture_output=True,text=True).stdout)
print('=== nvlink (if any) ===')
print(subprocess.run(['bash','-lc','nvidia-smi nvlink -s 2>/dev/null | head -20 || echo none'],capture_output=True,text=True).stdout)
print('=== torch per-rank mem (from CUDA) ===')
print('see train.log allocator stats below')
print('=== throughput from log (last 12 step times) ===')
log=open('/tmp/train.log').read()
its=re.findall(r'(\d+)/429 \[[^\]]*?([0-9.]+)s/it\]', log)
last=its[-12:]
if last:
    sps=[float(s) for _,s in last]
    print('recent s/it:', [f"{x:.2f}" for x in sps], '| mean', f"{st.mean(sps):.2f}s/it")
    cur=int(last[-1][0]); rem=(429-cur)*st.mean(sps)
    print(f'step {cur}/429 | ETA ~{rem/60:.1f} min')
losses=re.findall(r"'loss': '([0-9.]+)'.*?'epoch': '([0-9.]+)'", log)
print('loss history:', [f'{l}@e{e}' for l,e in losses[-8:]])

In [ ]:
import re
log=open('/tmp/train.log').read()
cur=re.findall(r'(\d+)/429 \[',log)
print('current step:', cur[-1] if cur else '?','/429')
print('train loss:', re.findall(r"'loss': '([0-9.eE-]+)'",log)[-8:])
print('eval_loss :', re.findall(r"'eval_loss': '([0-9.eE-]+)'",log) or 'none yet')
import subprocess
print(subprocess.run(['bash','-lc',"grep -iE 'eval_loss|early|best|stopping' /tmp/train.log | tail -12"],capture_output=True,text=True).stdout or 'no eval/early-stop lines yet')

In [ ]:
import subprocess, os, time
# graceful stop: kill torchrun launcher -> it SIGTERMs workers
print(subprocess.run(['bash','-lc',"pkill -TERM -f 'torch.distributed.run.*run_train_patched' ; sleep 3 ; pkill -TERM -f run_train_patched ; sleep 2 ; pkill -KILL -f run_train_patched ; echo killed"],capture_output=True,text=True).stdout)
time.sleep(2)
print('=== procs after kill ===')
print(subprocess.run(['bash','-lc',"ps -eo pid,cmd | grep -E 'run_train_patched|distributed.run' | grep -v grep"],capture_output=True,text=True).stdout or 'none (stopped)')
OUT='/tmp/calm_gemma4_12b_it_adapter'
print('=== checkpoints in', OUT, '===')
if os.path.isdir(OUT):
    for d in sorted(os.listdir(OUT)):
        p=os.path.join(OUT,d)
        if os.path.isdir(p):
            sz=sum(os.path.getsize(os.path.join(p,f)) for f in os.listdir(p) if os.path.isfile(os.path.join(p,f)))
            print(f'  {d:20s} {sz/1e6:8.1f} MB  files={os.listdir(p)[:6]}')
        else:
            print('  (file)',d)
else:
    print('OUT_DIR chua ton tai')